In [1]:
import pandas as pd
import numpy as np
import time
from nba_api.stats.endpoints import leaguegamelog

In [3]:
# Colect historic features for each player, to calculate its EWMA rating (a matric for impact of each player in the team performance)
def fetch_player_boxscores(seasons):
    """
    Colect logs of each player in the given seasons.
    seasons: list of strings, e.g., ['2015-16', '2016-17', '2017-18']
    """
    all_logs = []
    for season in seasons:
        print(f"Fetching player boxscores for season: {season}")
        log = leaguegamelog.LeagueGameLog(
            season=season,
            season_type_all_star='Regular Season',
            player_or_team_abbreviation='P'
        )
        df_season = log.get_data_frames()[0]
        all_logs.append(df_season)
        time.sleep(1.2)

    df = pd.concat(all_logs, ignore_index=True)
    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
    return df

def calculate_player_ewma_gmsc(df_player_logs, alpha=0.15):
    """
    Calculate the Exponentially Weighted Moving Average (EWMA) of Game Score (GMSC) for each player.
    alpha=0.15 amounts to a half-life of approximately ~4-5 games
    """
    df = df_player_logs.copy()

    # Official formula for GMSC (John Hollinger's Game Score) 
    df['GMSC'] = (
        df['PTS'] 
        + 0.4 * df['FGM'] 
        - 0.7 * df['FGA'] 
        - 0.4 * (df['FTA'] - df['FTM']) 
        + 0.7 * df['OREB'] 
        + 0.3 * df['DREB'] 
        + df['STL'] 
        + 0.7 * df['AST'] 
        + 0.7 * df['BLK'] 
        - 0.4 * df['PF'] 
        - df['TOV']
    )

    # Sort by player and game date
    df = df.sort_values(by=['PLAYER_ID', 'GAME_DATE', 'GAME_ID']).reset_index(drop=True)

    # EWMA calculation using shift(1) to avoid data leakage
    df['EWMA_GMSC_PRE_GAME'] = (
        df.groupby('PLAYER_ID')['GMSC']
        .transform(lambda s: s.ewm(alpha=alpha, min_periods=1).mean().shift(1))
    )
    
    df['AVG_MIN_PRE_GAME'] = (
        df.groupby('PLAYER_ID')['MIN']
        .transform(lambda s: s.ewm(alpha=alpha, min_periods=1).mean().shift(1))
    )
    
    return df

In [4]:
def export_current_player_impact_dict(df_processed, output_path='dataframes/player_impact_current.parquet'):
    """
    Export the last EWMA GMSC and AVG_MIN for each player to a parquet file.
    """
    # Drop rows where EWMA_GMSC_PRE_GAME is NaN, as we cannot assess impact without it
    df_valid = df_processed.dropna(subset=['EWMA_GMSC_PRE_GAME']).copy()

    latest_player_state = (
        df_valid.sort_values('GAME_DATE')
        .groupby('PLAYER_ID', as_index=False)
        .last()
    )

    impact_df = latest_player_state[[
        'PLAYER_ID', 'PLAYER_NAME', 'TEAM_ABBREVIATION', 
        'EWMA_GMSC_PRE_GAME', 'AVG_MIN_PRE_GAME'
    ]].copy()

    impact_df.rename(columns={
        'EWMA_GMSC_PRE_GAME': 'CURRENT_EWMA_GMSC',
        'AVG_MIN_PRE_GAME': 'CURRENT_AVG_MIN'
    }, inplace=True)
    
    impact_df.to_parquet(output_path, index=False)
    print(f"Saved dictonary containing {len(impact_df)} players to {output_path}")
    return impact_df

In [5]:
seasons_to_fetch = ['2024-25', '2025-26']
df_players = fetch_player_boxscores(seasons_to_fetch)

df_ewma = calculate_player_ewma_gmsc(df_players, alpha=0.15)

export_current_player_impact_dict(df_ewma, output_path='../dataframes/player_impact_current.parquet')

Fetching player boxscores for season: 2024-25
Fetching player boxscores for season: 2025-26
Saved dictonary containing 682 players to ../dataframes/player_impact_current.parquet


,PLAYER_ID,PLAYER_NAME,TEAM_ABBREVIATION,CURRENT_EWMA_GMSC,CURRENT_AVG_MIN
0,2544,LeBron James,LAL,21.048285,33.376804
1,101108,Chris Paul,LAC,3.019632,15.255069
2,200768,Kyle Lowry,PHI,0.741489,9.610007
3,200782,P.J. Tucker,NYK,2.054054,15.513514
4,201142,Kevin Durant,HOU,20.848203,35.879421
...,...,...,...,...,...
677,1643133,Lawson Lovering,MEM,3.400000,18.000000
678,1643141,Jahmyl Telfort,LAC,-0.363602,4.090873
679,1643158,Andersson Garcia,UTA,5.701969,30.394422
680,1643253,Toby Okani,MEM,5.159660,35.852931


In [2]:
from datetime import datetime
from pathlib import Path
import sys

project_root = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "src" / "injury_service.py").exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError("Could not locate src/injury_service.py.")

sys.path.insert(0, str(project_root))

from src.injury_service import get_nba_injuries_espn

# Testa para os jogos de hoje
injuries_today = get_nba_injuries_espn(datetime.now())

print(f"Times com registros de lesão encontrados: {len(injuries_today)}\n")
for team, list_inj in list(injuries_today.items())[:5]: # Exibe os 5 primeiros
    print(f"--- {team} ---")
    for p in list_inj:
        print(f"  • {p['name']} | Status: {p['status']} | Chance de Fora: {p['p_out']:.0%} | Motivo: {p['detail']}")

Times com registros de lesão encontrados: 6

--- DET ---
--- BOS ---
--- NYK ---
--- PHI ---
--- SAS ---
